# 01 - RadGraph-XL Full-Dataset Audit

This notebook is the canonical structural audit for the complete 2,300-report
RadGraph-XL collection: the 300-report MIMIC archive and the 2,000-report
Stanford JSONL file. It writes aggregate metadata only and never displays or
exports report tokens.

Original JSONL fields and audit-derived fields are reported separately. All
formal dissertation statistics must be taken from `outputs/full_2300/audit`;
the earlier `outputs/audit` directory is retained only as a 300-report archive.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import statistics
import zipfile
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

MIMIC_ZIP = Path(os.environ["RADGRAPH_XL_MIMIC_ZIP"])
STANFORD_JSONL = Path(os.environ["RADGRAPH_XL_STANFORD_JSONL"])
RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
if RUN_NAME != "full_2300":
    raise ValueError(
        "Notebook 01 is the canonical complete-dataset audit and requires "
        "RADGRAPH_XL_RUN_NAME=full_2300"
    )

AUDIT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_REPORTS = 2300
AUDIT_SCHEMA_VERSION = "2.0"
ORIGINAL_FIELDS = {"dataset", "doc_key", "sentences", "ner", "relations"}
AUDIT_DERIVED_FIELDS = {
    "source",
    "doc_id",
    "sentence_count",
    "token_count",
    "entity_count",
    "relation_count",
}

assert MIMIC_ZIP.exists(), f"MIMIC ZIP not found: {MIMIC_ZIP}"
assert STANFORD_JSONL.exists(), f"Stanford JSONL not found: {STANFORD_JSONL}"

print(
    {
        "run_name": RUN_NAME,
        "dataset_scope": "complete 2,300-report RadGraph-XL collection",
        "mimic_zip": str(MIMIC_ZIP),
        "stanford_jsonl": str(STANFORD_JSONL),
        "audit_dir": str(AUDIT_DIR),
    }
)


In [ ]:
def load_jsonl_lines(lines, source: str) -> tuple[list[dict], str]:
    records = []
    digest = hashlib.sha256()
    for line_number, raw_line in enumerate(lines, start=1):
        if not raw_line.strip():
            continue
        digest.update(raw_line)
        record = json.loads(raw_line)
        records.append({"source": source, "record": record, "line_number": line_number})
    return records, digest.hexdigest()


def load_mimic_zip(path: Path) -> tuple[list[dict], dict]:
    with zipfile.ZipFile(path) as archive:
        members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]
        if len(members) != 1:
            raise ValueError(f"Expected one JSONL member, found {members}")
        member = members[0]
        with archive.open(member) as handle:
            records, digest = load_jsonl_lines(handle, "mimic")
    return records, {
        "path": str(path),
        "jsonl_member": member,
        "jsonl_sha256": digest,
        "records": len(records),
    }


def load_stanford_jsonl(path: Path) -> tuple[list[dict], dict]:
    with path.open("rb") as handle:
        records, digest = load_jsonl_lines(handle, "stanford")
    return records, {
        "path": str(path),
        "jsonl_sha256": digest,
        "records": len(records),
    }


mimic_records, mimic_input = load_mimic_zip(MIMIC_ZIP)
stanford_records, stanford_input = load_stanford_jsonl(STANFORD_JSONL)
loaded_records = mimic_records + stanford_records

if len(loaded_records) != EXPECTED_REPORTS:
    raise ValueError(f"Expected {EXPECTED_REPORTS} reports, found {len(loaded_records)}")

print({"mimic": len(mimic_records), "stanford": len(stanford_records), "combined": len(loaded_records)})


In [ ]:
validation_errors = Counter()
top_level_shapes = Counter()
doc_key_types = Counter()
dataset_counts = Counter()
source_counts = Counter()
entity_labels = Counter()
relation_labels = Counter()
report_rows = []
doc_ids = []

for item in loaded_records:
    source = item["source"]
    record = item["record"]
    fields = set(record)
    top_level_shapes[tuple(sorted(fields))] += 1
    if fields != ORIGINAL_FIELDS:
        validation_errors["unexpected_top_level_fields"] += 1

    dataset = str(record.get("dataset"))
    source_doc_key = str(record.get("doc_key"))
    doc_id = f"{dataset}::{source_doc_key}"
    doc_ids.append(doc_id)
    doc_key_types[type(record.get("doc_key")).__name__] += 1

    sentences = record.get("sentences")
    ner = record.get("ner")
    relations = record.get("relations")
    if not isinstance(sentences, list) or not all(isinstance(sentence, list) for sentence in sentences):
        validation_errors["sentences_not_nested_lists"] += 1
        continue
    if not isinstance(ner, list) or not isinstance(relations, list):
        validation_errors["annotations_not_lists"] += 1
        continue
    if not (len(sentences) == len(ner) == len(relations)):
        validation_errors["outer_list_alignment"] += 1

    token_count = sum(len(sentence) for sentence in sentences)
    entity_count = sum(len(sentence_entities) for sentence_entities in ner)
    relation_count = sum(len(sentence_relations) for sentence_relations in relations)

    source_counts[source] += 1
    dataset_counts[dataset] += 1
    for sentence_entities in ner:
        entity_labels.update(str(annotation[2]) for annotation in sentence_entities)
    for sentence_relations in relations:
        relation_labels.update(str(annotation[4]) for annotation in sentence_relations)

    report_rows.append(
        {
            "source": source,
            "doc_id": doc_id,
            "source_doc_key": source_doc_key,
            "dataset": dataset,
            "sentence_count": len(sentences),
            "token_count": token_count,
            "entity_count": entity_count,
            "relation_count": relation_count,
        }
    )

duplicate_doc_ids = len(doc_ids) - len(set(doc_ids))
if duplicate_doc_ids:
    validation_errors["duplicate_dataset_doc_key"] += duplicate_doc_ids

if validation_errors:
    raise ValueError(f"Dataset validation failed: {dict(validation_errors)}")

print(
    {
        "validated_reports": len(report_rows),
        "top_level_fields": sorted(ORIGINAL_FIELDS),
        "doc_key_types": dict(doc_key_types),
        "duplicate_dataset_doc_keys": duplicate_doc_ids,
    }
)


In [ ]:
def distribution(values: list[int]) -> dict:
    ordered = sorted(values)
    return {
        "min": min(ordered),
        "median": statistics.median(ordered),
        "mean": round(statistics.mean(ordered), 2),
        "p95": ordered[min(len(ordered) - 1, int(0.95 * len(ordered)))],
        "max": max(ordered),
    }


token_counts = [row["token_count"] for row in report_rows]
audit = {
    "audit_schema_version": AUDIT_SCHEMA_VERSION,
    "generated_by": "notebooks/01_data_audit.ipynb",
    "run_name": RUN_NAME,
    "canonical_output": True,
    "dataset_scope": "complete RadGraph-XL collection",
    "inputs": {"mimic": mimic_input, "stanford": stanford_input},
    "original_dataset_fields": sorted(ORIGINAL_FIELDS),
    "audit_derived_summary_fields": sorted(AUDIT_DERIVED_FIELDS),
    "reports": len(report_rows),
    "reports_by_source": dict(sorted(source_counts.items())),
    "reports_by_dataset": dict(sorted(dataset_counts.items())),
    "anatomy_modality_groups": sorted(dataset_counts),
    "tokens": sum(token_counts),
    "reports_exceeding_512_original_tokens": sum(value > 512 for value in token_counts),
    "entities": sum(row["entity_count"] for row in report_rows),
    "relations": sum(row["relation_count"] for row in report_rows),
    "entity_labels": dict(sorted(entity_labels.items())),
    "relation_labels": dict(sorted(relation_labels.items())),
    "report_token_distribution": distribution(token_counts),
    "report_entity_distribution": distribution([row["entity_count"] for row in report_rows]),
    "report_relation_distribution": distribution([row["relation_count"] for row in report_rows]),
    "indexing_convention": {
        "base": "zero-based",
        "end_index": "inclusive",
        "scope": "document-level token sequence",
    },
    "validation": {
        "top_level_shapes": {"|".join(shape): count for shape, count in top_level_shapes.items()},
        "doc_key_types": dict(doc_key_types),
        "duplicate_dataset_doc_keys": duplicate_doc_ids,
        "errors": dict(validation_errors),
    },
}

print(
    json.dumps(
        {
            "reports": audit["reports"],
            "reports_by_source": audit["reports_by_source"],
            "reports_by_dataset": audit["reports_by_dataset"],
            "tokens": audit["tokens"],
            "reports_exceeding_512_original_tokens": audit[
                "reports_exceeding_512_original_tokens"
            ],
            "entities": audit["entities"],
            "relations": audit["relations"],
        },
        indent=2,
    )
)


In [ ]:
audit_path = AUDIT_DIR / "data_audit.json"
summary_path = AUDIT_DIR / "report_summary.csv"
manifest_path = AUDIT_DIR / "audit_manifest.json"

audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(report_rows[0]))
    writer.writeheader()
    writer.writerows(report_rows)

manifest = {
    "canonical_dataset_scope": "complete 2,300-report RadGraph-XL collection",
    "run_name": RUN_NAME,
    "notebook": "notebooks/01_data_audit.ipynb",
    "contains_report_text": False,
    "outputs": {
        "data_audit": str(audit_path),
        "report_summary": str(summary_path),
    },
    "historical_300_report_outputs": str(PROJECT_ROOT / "outputs" / "audit"),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Wrote canonical aggregate-only outputs:")
print(" -", audit_path)
print(" -", summary_path)
print(" -", manifest_path)
